### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [ ]:
!pip install langchain

In [5]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [1]:
!pip install langchain_groq

In [6]:
a=5


In [7]:
print(a)

5


In [14]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C0A7F71950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C0A7F72850>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
!pip install langchain_core

In [24]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to whatever it is"),
    HumanMessage(content="'baka oyasumi itterashayi'")
]

result=model.invoke(messages)
result

AIMessage(content='**Language:** Japanese (though the phrase is a bit garbled / colloquial)\n\n**Literal breakdown**\n\n| Word | Kana / Kanji | Rough meaning |\n|------|--------------|----------------|\n| **baka** | ばか | “idiot”, “fool”, “stupid” (often used as an insult, but can be teasing) |\n| **oyasumi** | おやすみ | “good night” (short for *oyasumi nasai*) |\n| **itterashayi** | いってらっしゃい / いってらっしょい (dialectal) | “go and come back”, “see you later”, “take care” – a casual farewell said to someone who’s leaving. In some dialects it can be pronounced *itterasshoi* or *itterashii* and written as いってらっしょい. |\n\n**Putting it together**\n\nThe phrase is essentially a string of three separate expressions, not a single grammatical sentence:\n\n> **“Baka, oyasumi, itterashayi!”**  \n\n≈ “You idiot, good night, see you later!”  \n\nor, in a more natural English tone:\n\n> “Hey, you fool – good night, and take care!”  \n\nBecause the three parts are spoken one after another, the overall feeling i

In [25]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'**Language:** Japanese (though the phrase is a bit garbled / colloquial)\n\n**Literal breakdown**\n\n| Word | Kana / Kanji | Rough meaning |\n|------|--------------|----------------|\n| **baka** | ばか | “idiot”, “fool”, “stupid” (often used as an insult, but can be teasing) |\n| **oyasumi** | おやすみ | “good night” (short for *oyasumi nasai*) |\n| **itterashayi** | いってらっしゃい / いってらっしょい (dialectal) | “go and come back”, “see you later”, “take care” – a casual farewell said to someone who’s leaving. In some dialects it can be pronounced *itterasshoi* or *itterashii* and written as いってらっしょい. |\n\n**Putting it together**\n\nThe phrase is essentially a string of three separate expressions, not a single grammatical sentence:\n\n> **“Baka, oyasumi, itterashayi!”**  \n\n≈ “You idiot, good night, see you later!”  \n\nor, in a more natural English tone:\n\n> “Hey, you fool – good night, and take care!”  \n\nBecause the three parts are spoken one after another, the overall feeling is casual, slightly

In [26]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'**Language:** Japanese (the phrase is a mix of colloquial Japanese words)\n\n**Rough English translation**\n\n> “You idiot, good night – go away!”  \n\n**Break‑down of the parts**\n\n| Japanese word | Literal meaning / note | How it contributes to the whole |\n|---------------|------------------------|---------------------------------|\n| **baka** (ばか) | “idiot”, “fool”, “stupid” | An insult or teasing address. |\n| **oyasumi** (おやすみ) | “good night” (short for *oyasumi nasai*) | A farewell used at night. |\n| **itterashayi** (いってらっしゃい) – written here phonetically as *itterashayi* | “go and come back”, “take care”, “see you later” (often said when someone is leaving) | In this context it’s used sarcastically to tell the person to go away. |\n\nPutting the three parts together gives the colloquial, slightly mocking line “You idiot, good night – go away!” (or “Stupid, good night, get out!”).  \n\n*Note:* Because the original string is not standard Japanese spelling (it mixes casual forms

In [20]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [22]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [23]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into French:'),
 HumanMessage(content='Hello')]

In [24]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour \n'

In [25]:
!pip install streamlit